In [7]:
import cv2
import numpy as np

In [13]:
import cv2

# 1a: Cargar una imagen y mostrar el valor RGB del píxel bajo el cursor

ruta_imagen = "Imagen1.jpg"
img = cv2.imread(ruta_imagen)

if img is None:
    raise FileNotFoundError(f"No se pudo cargar la imagen en: {ruta_imagen}")

# --- Escalar la imagen para que quepa en pantalla sin recortarse ---
ANCHO_MAX, ALTO_MAX = 1280, 720
alto_original, ancho_original = img.shape[:2]

escala = min(ANCHO_MAX / ancho_original, ALTO_MAX / alto_original, 1.0)
ancho_mostrado = int(ancho_original * escala)
alto_mostrado = int(alto_original * escala)

img_mostrada = cv2.resize(img, (ancho_mostrado, alto_mostrado), interpolation=cv2.INTER_AREA)

ventana = "Practica 1a - Valor RGB bajo el cursor"
cv2.namedWindow(ventana)


def mostrar_rgb(event, x, y, flags, param):
    if event == cv2.EVENT_MOUSEMOVE:
        # Convertimos la posición del ratón (sobre la imagen escalada)
        # a coordenadas de la imagen original, para leer el píxel real
        x_original = min(int(x / escala), ancho_original - 1)
        y_original = min(int(y / escala), alto_original - 1)

        # OpenCV almacena los píxeles en formato BGR
        b, g, r = img[y_original, x_original]
        texto = f"RGB: ({r}, {g}, {b})  Pos: ({x_original}, {y_original})"

        vista = img_mostrada.copy()
        cv2.putText(vista, texto, (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.imshow(ventana, vista)


cv2.setMouseCallback(ventana, mostrar_rgb)
cv2.imshow(ventana, img_mostrada)

print("Mueve el ratón sobre la imagen para ver el valor RGB del píxel.")
print("Pulsa 'q' o ESC para salir.")

while True:
    key = cv2.waitKey(20) & 0xFF
    if key == ord('q') or key == 27:
        break

cv2.destroyAllWindows()

Mueve el ratón sobre la imagen para ver el valor RGB del píxel.
Pulsa 'q' o ESC para salir.


In [17]:
import cv2
import numpy as np

# --- 1b: Dibujar líneas, rectángulos y círculos con el ratón ---
# El color se especifica en RGB (mediante 3 trackbars) y el grosor con otra trackbar.

ANCHO, ALTO = 800, 600
lienzo = np.ones((ALTO, ANCHO, 3), dtype=np.uint8) * 255

ventana = "Practica 1b - Dibujo con el raton"
cv2.namedWindow(ventana)


def nada(x):
    pass


cv2.createTrackbar("R", ventana, 0, 255, nada)
cv2.createTrackbar("G", ventana, 0, 255, nada)
cv2.createTrackbar("B", ventana, 255, 255, nada)
cv2.createTrackbar("Grosor", ventana, 2, 30, nada)

modo = "linea"  # 'linea', 'rectangulo' o 'circulo'
dibujando = False
pt_inicio = (0, 0)
lienzo_temporal = lienzo.copy()


def color_actual():
    r = cv2.getTrackbarPos("R", ventana)
    g = cv2.getTrackbarPos("G", ventana)
    b = cv2.getTrackbarPos("B", ventana)
    return (b, g, r)  # OpenCV dibuja en BGR


def grosor_actual():
    return max(1, cv2.getTrackbarPos("Grosor", ventana))


def dibujar_primitiva(destino, x, y):
    color = color_actual()
    grosor = grosor_actual()
    if modo == "linea":
        cv2.line(destino, pt_inicio, (x, y), color, grosor)
    elif modo == "rectangulo":
        cv2.rectangle(destino, pt_inicio, (x, y), color, grosor)
    elif modo == "circulo":
        radio = int(np.hypot(x - pt_inicio[0], y - pt_inicio[1]))
        cv2.circle(destino, pt_inicio, radio, color, grosor)


def evento_raton(event, x, y, flags, param):
    global dibujando, pt_inicio, lienzo, lienzo_temporal

    if event == cv2.EVENT_LBUTTONDOWN:
        dibujando = True
        pt_inicio = (x, y)
        lienzo_temporal = lienzo.copy()

    elif event == cv2.EVENT_MOUSEMOVE and dibujando:
        vista = lienzo_temporal.copy()
        dibujar_primitiva(vista, x, y)
        cv2.imshow(ventana, vista)

    elif event == cv2.EVENT_LBUTTONUP:
        dibujando = False
        dibujar_primitiva(lienzo, x, y)
        cv2.imshow(ventana, lienzo)


cv2.setMouseCallback(ventana, evento_raton)
cv2.imshow(ventana, lienzo)

print("Controles:")
print("  l -> modo línea")
print("  r -> modo rectángulo")
print("  c -> modo círculo")
print("  Ajusta R, G, B y Grosor con las barras deslizantes")
print("  n -> limpiar el lienzo")
print("  q / ESC -> salir")

while True:
    key = cv2.waitKey(20) & 0xFF
    if key == ord('q') or key == 27:
        break
    elif key == ord('l'):
        modo = "linea"
        print("Modo: línea")
    elif key == ord('r'):
        modo = "rectangulo"
        print("Modo: rectángulo")
    elif key == ord('c'):
        modo = "circulo"
        print("Modo: círculo")
    elif key == ord('n'):
        lienzo[:] = 255
        lienzo_temporal = lienzo.copy()
        cv2.imshow(ventana, lienzo)

cv2.destroyAllWindows()

Controles:
  l -> modo línea
  r -> modo rectángulo
  c -> modo círculo
  Ajusta R, G, B y Grosor con las barras deslizantes
  n -> limpiar el lienzo
  q / ESC -> salir
Modo: rectángulo
Modo: rectángulo
Modo: círculo
